**04 | Subject-Independent Data Splitting**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Apply existing HARTH loading, windowing and feature extraction to **eight different participants**, then create reproducible, strictly participant-disjoint Train/Validation/Test partitions. This is an educational multi-person subset, **not** the full benchmark. No model fitting is performed.

**01. Imports and reusable Python functions**

Code comments identify the role of each operation. The algorithms remain in dedicated `.py` modules.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the repository and import the reusable functions.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root: raise RuntimeError("Not inside the project")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features
from core.data_splitting import split_group_indices
from harth_data import ACTIVITY_NAMES, ensure_harth_sample, load_harth_subject
print("Shared Python functions: ready")

Shared Python functions: ready


**02. Acquire a bounded, multi-participant HARTH subset**

Only selected v2.0 files are downloaded. Together the eight files require approximately 124 MiB. Files are kept under ignored `data/raw/`; large datasets are not committed.

In [2]:
# Select source-verified independent subjects.
SUBJECTS = ["S023", "S024", "S025", "S027", "S028", "S032", "S034", "S035"]
SOURCE = root / "data/raw/harth_v2"
paths = {s: ensure_harth_sample(SOURCE, subject=s) for s in SUBJECTS}
display(pd.DataFrame([
    {"subject": s, "MiB": round(paths[s].stat().st_size/1048576,2)}
    for s in SUBJECTS
]))

Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S023.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S024.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S025.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S027.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S028.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S032.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S034.csv
Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S035.csv


,subject,MiB
0,S023,11.50
1,S024,15.95
2,S025,20.69
3,S027,15.14
4,S028,16.16
5,S032,14.13
6,S034,14.34
7,S035,14.21


**03. Window and extract features separately for each person**

The same 2-second windows and five feature methods from Notebooks 02 and 03 are used. This guarantees windows never cross participant boundaries.

**Label provenance note:** In the real `S032.csv`, codes `10` and `11` appear but are not defined in the authors' HARTH v2.0 activity table. For this inspection-only notebook, they remain unchanged and are explicitly flagged. Neither code is assigned an invented activity meaning.

In [3]:
feature_blocks, metadata_blocks, subjects_summary = [], [], []
for subject in SUBJECTS:
    # Process each participant independently before combining rows.
    # Retain source-unknown activity codes only for the metadata audit.
    data, channels = load_harth_subject(
        paths[subject], allow_unmapped_labels=True
    )
    unknown_mask = ~data["label"].isin(ACTIVITY_NAMES)
    unknown_codes = sorted(data.loc[unknown_mask, "label"].unique().tolist())
    if unknown_codes:
        print(f"{subject}: undocumented HARTH codes = {unknown_codes}")
    X, y, info = create_signal_windows(
        data[channels].to_numpy(), data["label"].to_numpy(),
        window_size=100, step_size=50,
        timestamps=data["timestamp"], sampling_rate=50,
        min_label_purity=1.0, subject_id=subject,
    )
    if not len(X): raise ValueError(f"No valid windows for {subject}")
    features = extract_signal_features(X, channel_names=channels)
    feature_blocks.append(features)
    metadata_blocks.append(info)
    subjects_summary.append({"subject":subject,"raw_samples":len(data),
                             "valid_windows":len(y),"classes":len(np.unique(y)),
                             "unmapped_rows":int(unknown_mask.sum()),
                             "unmapped_codes":str(unknown_codes)})
# Preserve identical row ordering in feature and metadata tables.
feature_table = pd.concat(feature_blocks, ignore_index=True)
window_metadata = pd.concat(metadata_blocks, ignore_index=True)
assert len(feature_table) == len(window_metadata)
display(pd.DataFrame(subjects_summary))
print("Combined feature matrix:",feature_table.shape)

HARTH subject S023: 137,646 rows, 6 sensors
Input signal: 137,646 samples, 6 channels
Candidate windows: 2,751
Retained windows: 2,567 | Shape: (2567, 6, 100)
Rejected candidates: 184
Input windows: (2567, 6, 100)
Feature matrix: (2567, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S024: 170,534 rows, 6 sensors
Input signal: 170,534 samples, 6 channels
Candidate windows: 3,409
Retained windows: 3,119 | Shape: (3119, 6, 100)
Rejected candidates: 290
Input windows: (3119, 6, 100)
Feature matrix: (3119, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S025: 231,729 rows, 6 sensors
Input signal: 231,729 samples, 6 channels
Candidate windows: 4,633
Retained windows: 4,272 | Shape: (4272, 6, 100)
Rejected candidates: 361
Input windows: (4272, 6, 100)
Feature matrix: (4272, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S027: 158,584 rows, 6 sensors
Input signal: 158,584 samples, 6 channels
Candidate windows: 3

C:\Users\Reza\AppData\Local\Temp\ipykernel_10988\67100086.py:5: UserWarning: Unexpected HARTH activity codes: [10, 11] (2,018 rows in S032.csv). The HARTH v2.0 activity table does not define these codes. Source codes retained without reinterpretation; resolve them before supervised training.
  data, channels = load_harth_subject(


HARTH subject S032: 178,341 rows, 6 sensors
S032: undocumented HARTH codes = [10, 11]
Input signal: 178,341 samples, 6 channels
Candidate windows: 3,565
Retained windows: 3,038 | Shape: (3038, 6, 100)
Rejected candidates: 527
Input windows: (3038, 6, 100)
Feature matrix: (3038, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S034: 188,202 rows, 6 sensors
Input signal: 188,202 samples, 6 channels
Candidate windows: 3,763
Retained windows: 3,351 | Shape: (3351, 6, 100)
Rejected candidates: 412
Input windows: (3351, 6, 100)
Feature matrix: (3351, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S035: 181,251 rows, 6 sensors
Input signal: 181,251 samples, 6 channels
Candidate windows: 3,624
Retained windows: 3,337 | Shape: (3337, 6, 100)
Rejected candidates: 287
Input windows: (3337, 6, 100)
Feature matrix: (3337, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length


,subject,raw_samples,valid_windows,classes,unmapped_rows,unmapped_codes
0,S023,137646,2567,8,0,[]
1,S024,170534,3119,8,0,[]
2,S025,231729,4272,12,0,[]
3,S027,158584,3082,6,0,[]
4,S028,165178,3197,7,0,[]
5,S032,178341,3038,7,2018,"[10, 11]"
6,S034,188202,3351,7,0,[]
7,S035,181251,3337,5,0,[]


Combined feature matrix: (25963, 30)


**04. Split participant IDs, not overlapping windows**

Target allocation is 4 Train, 2 Validation and 2 Test individuals. Fractions apply to numbers of participants, not numbers of windows.

In [4]:
# Use a deterministic random seed and allocate whole subjects.
partitions = split_group_indices(
    window_metadata["subject_id"].to_numpy(),
    train_fraction=0.50, validation_fraction=0.25,
    test_fraction=0.25, random_state=42,
)
groups = {
    name: set(window_metadata.iloc[index]["subject_id"])
    for name,index in partitions.items()
}
# Verify complete partition coverage and absence of identity leakage.
assert not (groups["train"]&groups["validation"] or
            groups["train"]&groups["test"] or groups["validation"]&groups["test"])
np.testing.assert_array_equal(
    np.sort(np.concatenate(list(partitions.values()))),
    np.arange(len(window_metadata))
)
display(pd.DataFrame([
    {"partition":p,"subjects":len(groups[p]),"windows":len(ix),
     "IDs":", ".join(sorted(groups[p]))}
    for p,ix in partitions.items()
]))
print("Subject leakage: NONE")

Independent groups: 8
train: 4 groups | 13,888 rows
validation: 2 groups | 6,470 rows
test: 2 groups | 5,605 rows


,partition,subjects,windows,IDs
0,train,4,13888,"S025, S027, S028, S035"
1,validation,2,6470,"S024, S034"
2,test,2,5605,"S023, S032"


Subject leakage: NONE


**05. Compare activity distributions across Train, Validation and Test**

Grouped splitting prevents overlap between participants, but it does **not** guarantee balanced activity classes. The following **single Figure contains two complementary panels**:

- **Grouped bar chart (top):** each activity has its own bar beside the other activities within Train, Validation and Test. The bars are **not stacked**.
- **Heatmap (bottom):** the same within-partition percentages, with activities as rows and Train/Validation/Test as columns. Every cell shows its exact percentage.

Both panels use percentages rather than raw counts so that partitions with different numbers of windows can be compared. Undocumented source label codes remain explicitly marked; this visual audit is **not** permission to train a classifier on those codes.

In [ ]:
# 1. Count activity windows in each subject-independent partition.
partition_order = ["train", "validation", "test"]
counts = pd.DataFrame({
    part: window_metadata.iloc[partitions[part]]["label"].value_counts()
    for part in partition_order
}).fillna(0).astype(int).sort_index()
counts.index = [
    f"{int(code)}: {ACTIVITY_NAMES.get(int(code), 'UNDOCUMENTED source code')}"
    for code in counts.index
]
display(counts)

# 2. Express each activity as a percentage of its own partition.
#    This avoids comparing raw counts when partitions have different sizes.
fractions = counts.div(counts.sum(axis=0), axis=1) * 100
activity_names = fractions.index.tolist()
n_activities = len(activity_names)
positions = np.arange(len(partition_order))
bar_width = 0.82 / n_activities
colours = plt.get_cmap("tab20")(np.linspace(0, 1, n_activities))

# 3. Create ONE figure containing a grouped bar chart and a heatmap.
fig, (ax_bars, ax_heatmap) = plt.subplots(
    2, 1, figsize=(18, 12),
    gridspec_kw={"height_ratios": [1.15, 1.35]},
)

# 4. Group the activity bars side by side under Train, Validation and Test.
#    Unlike a stacked bar chart, each bar starts at zero.
for i, (activity, colour) in enumerate(zip(activity_names, colours)):
    x_positions = positions + (i - (n_activities - 1) / 2) * bar_width
    ax_bars.bar(
        x_positions,
        fractions.loc[activity, partition_order].to_numpy(),
        width=bar_width * 0.92,
        label=activity,
        color=colour,
    )

ax_bars.set_xticks(positions, ["Train", "Validation", "Test"])
ax_bars.set(
    title="HARTH: activity proportions by data partition (grouped bars)",
    xlabel="Partition",
    ylabel="Fraction of windows (%)",
    ylim=(0, 100),
)
ax_bars.grid(axis="y", alpha=0.25)
ax_bars.set_axisbelow(True)
ax_bars.legend(
    title="Activity",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
    frameon=False,
)

# 5. Use the SAME percentages in a heatmap for accurate comparisons.
#    Rows are activities; columns are Train, Validation and Test.
heat_values = fractions[partition_order].to_numpy()
heatmap = ax_heatmap.imshow(
    heat_values,
    aspect="auto",
    cmap="YlGnBu",
    vmin=0,
    vmax=max(1.0, float(heat_values.max())),
)
ax_heatmap.set_xticks(positions, ["Train", "Validation", "Test"])
ax_heatmap.set_yticks(np.arange(n_activities), activity_names)
ax_heatmap.set(title="HARTH: activity proportions (heatmap)", xlabel="Partition")

# 6. Show the exact percentage inside each heatmap cell.
for row in range(n_activities):
    for column in range(len(partition_order)):
        value = heat_values[row, column]
        text_colour = "white" if value > heat_values.max() * 0.55 else "black"
        ax_heatmap.text(
            column, row, f"{value:.1f}%",
            ha="center", va="center", fontsize=8, color=text_colour,
        )

colourbar = fig.colorbar(heatmap, ax=ax_heatmap, fraction=0.035, pad=0.025)
colourbar.set_label("Fraction of windows (%)")
fig.subplots_adjust(left=0.25, right=0.78, top=0.95, bottom=0.06, hspace=0.36)
plt.show()

# 7. Report missing classes and unresolved source annotation codes.
print("Classes absent by partition:", {
    part: counts.index[counts[part] == 0].tolist()
    for part in partition_order
})
unmapped_window_codes = sorted(
    set(window_metadata["label"].unique()) - set(ACTIVITY_NAMES)
)
print("Undocumented codes in retained windows:", unmapped_window_codes)
print("Classification with undocumented codes: NOT APPROVED")


**06. Final quality gate**

No model fitting or learned preprocessing has occurred. Train-only scaling and a classification baseline require careful review of class coverage first.

In [6]:
# Check deterministic features, complete grouping and valid shapes.
assert feature_table.shape[1] == 30
assert np.isfinite(feature_table.to_numpy()).all()
assert window_metadata["subject_id"].nunique() == 8
assert len(feature_table) == sum(len(ix) for ix in partitions.values())
for part,ix in partitions.items():
    print(f"{part}: X={feature_table.iloc[ix].shape}, y={window_metadata.iloc[ix]['label'].shape}")
print("Subject-independent splitting: PASSED")
print("Feature scaling trained: NO | Classifier trained: NO")
# The group split can be checked, but annotation provenance remains unresolved.
print("Undocumented label review required:", bool(unmapped_window_codes))


train: X=(13888, 30), y=(13888,)
validation: X=(6470, 30), y=(6470,)
test: X=(5605, 30), y=(5605,)
Subject-independent splitting: PASSED
Feature scaling trained: NO | Classifier trained: NO
Undocumented label review required: True
